# Exploratory analysis: bounded raw prefixes

Read-only samples. Each source is limited to 20,000 rows; Yoochoose uses separate click/buy limits. These are source prefixes, not random samples and not full-data statistics. They cannot support conclusions about model superiority. Timestamps are UTC seconds; equal seconds are simultaneous. See `docs/raw_data_structure.md` for full-source checks and sampling limitations.

In [ ]:
from pathlib import Path
import sys
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/main_config.yaml").is_file())
sys.path.insert(0, str(project_root))
import polars as pl
from src.config.loader import load_config, DATASETS
from src.generator.adapters.registry import make_adapter
config = load_config(project_root / "configs")
samples = {name: make_adapter(config.dataset(name), row_limit=20_000).normalize(max_rows=40_000) for name in DATASETS}


In [ ]:
summary = pl.DataFrame([{"dataset": name, "sample_events": frame.height, "sample_users": frame["user_id"].n_unique(), "sample_items": frame["item_id"].n_unique(), "min_UTC_second": frame["timestamp"].min(), "max_UTC_second": frame["timestamp"].max()} for name, frame in samples.items()])
display(summary)
label_counts = pl.concat([frame.group_by("label").len().with_columns(pl.lit(name).alias("dataset")) for name, frame in samples.items()]).sort(["dataset", "label"])
display(label_counts)


Labels have dataset-specific meanings: RetailRocket transaction, Taobao buy, Yoochoose click, MovieLens rating >=4 define the primary targets. In particular, a Yoochoose purchase is not the click target. Prefix counts below are descriptive only.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
figure, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for axis, name in zip(axes, DATASETS):
    table = samples[name].group_by("label").len().sort("label")
    axis.bar(table["label"].to_numpy(), table["len"].to_numpy(), color="#2878b5")
    axis.set(title=name, xlabel="Dataset label", xticks=[0, 1, 2])
axes[0].set_ylabel("Events in raw prefix")
figure.suptitle("Bounded source prefixes — not full dataset statistics")
display(figure)
plt.close(figure)


The production generator uses complete histories and fixed temporal boundaries. A prefix sample can miss validation/test periods, so it is never assigned invented split dates. Taobao category metadata defaults to observed-as-of after the full-source consistency check rejected a static mapping. MovieLens genome scores remain disabled in strict runs.